Complete the exercises below For **Assignment #4**.

Use **Markdown** cells for the non-code answers.

In this assignment we will work with the data underlying the *FiveThirtyEight* article ["Higher Rates Of Hate Crimes Are Tied To Income Inequality"](https://fivethirtyeight.com/features/higher-rates-of-hate-crimes-are-tied-to-income-inequality/).

Load the `tidymodels`, `readr`, and `moderndive` packages in the cell below.

In [ ]:
# I COULD NOT GET BINDER TO RUN, AND I GOT MESSAGES ON CODESPACES ABOUT BEING OVER MY MONTHLY USAGE LIMIT
## SO I CODED THIS UP IN R, USED ANACONDA TO MANIPULATE THE CELLS, AND COPY AND PASTE RAW R CODE AT BOTTOM OF THIS DOCUMENT
## THESE SPIFFY NEW CODING TOOLS ARE SUPER FRAGILE, UNFORTUNATELY!

library(tidymodels)
library(readr)
library(moderndive)



We can read the data from a **CSV file** at the following URL: [http://bit.ly/2ItxYg3](http://bit.ly/2ItxYg3).

Use the `read_csv` function to read the data into our R session. Call the new table `hate_crimes`.

In [ ]:
library(curl)
hate_crimes = read_csv('http://bit.ly/2ItxYg3')
head(hate_crimes)


Next, let’s add the high-school degree variable into the mix by creating a scatterplot showing:

- Income on the y-axis (this is the `income` variable)
- Percent of adults 25 or older with a high school degree on the x-axis (this is the `hs` variable)
- The points colored by level of urbanization in a region (this is the variable `urbanization`)


**In addition, add a line of best fit (regression line) for each level of the variable urbanization (one for “low”, one for “high”).**

*Add the regression lines to the plot using the `geom_parallel_slopes` function from the `moderndive` package. This function will draw the regression lines based on fitting a regression model with parallel slopes (i.e., with no interaction between `hs` and `urbanization`).*

In [ ]:

gg = ggplot(data=hate_crimes, aes(x=hs,y=income, group=urbanization)) + geom_point(aes(col=share_pop_metro, shape=urbanization)) + 
  geom_parallel_slopes(show.legend=TRUE) 
gg



❓Which regression line (high urbanization or low urbanization) appears to have the larger intercept?

**Answer:**

## The high household income category seems to have a higher intercept.  
## I added a different symbol for the points so I could be sure which is urbanization high and low.



Now let’s create a second scatterplot using the same variables, but this time draw the regression lines using `geom_smooth(method = "lm")`, which will allow for separate, non-parallel slopes for each urbanization group. 

**Code your scatter plot in the cell below.**

In [ ]:
gg2 = ggplot(data=hate_crimes, aes(x=hs,y=income, group=urbanization, col=share_pop_metro, shape=urbanization)) + geom_point() + 
 geom_smooth(method='lm')    
gg2



❓Based on visually comparing the two models shown above, do you think it would be best to run a “parallel slopes” model (i.e. a model that estimates one shared slope for the two levels of urbanization), or a more complex “interaction model” (i.e. a model that estimates a separate slope for the two levels of urbanization)?

**Answer:**


# The models look very similar visually, so I'd go with no separate slope as that is the simpler model.



Fit the following two regression models that examine the relationship between household `income` (as response variable), and high-school education (`hs`) and `urbanization` as explanatory variables:

1. A parallel slopes model (i.e., no interaction between `hs` and `urbanization`). ❗️Save the data recipe and model under the variables `ps_rec` and `ps_mod`, respectively. 
1. A non-parallel slopes model (i.e., allow `hs` and `urbanization` to interact in your model). ❗️Save the data recipe and model under the variable: `nps_rec` and `nps_mod`, respectively.

**Code you your models in the cell below.**

In [ ]:
# parallel slopes model

ps_rec = recipe(income ~ hs + urbanization, data = hate_crimes) |> 
    step_naomit(everything()) |>   # remove missing values
    step_dummy(urbanization) |>    # dummy encode the urbanization variable
    prep()                         # run the recipe on the training data provided

mod = linear_reg() |> set_engine('lm')

ps_mod = mod |> fit(income ~ ., juice(ps_rec))

ps_mod

In [ ]:
# non-parallel slopes model

# ❗️ you can use most of the code above, just add step_interact(~ starts_with("urbanization"):hs) to the recipe and
#    switch the variable names...


nps_rec = recipe(income ~ hs + urbanization , data = hate_crimes) |> 
    step_naomit(everything()) |>   # remove missing values
    step_dummy(urbanization) |>    # dummy encode the urbanization variable
    step_interact(terms = ~ starts_with("urbanization"):hs) |>
    prep()                         # run the recipe on the training data provided

nmod = linear_reg() |> set_engine('lm')

nps_mod = nmod |> fit(income ~ ., juice(nps_rec))

nps_mod





The following code creates a table of your model predictions over the training data. Calculate the [coefficient of determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) (R<sup>2</sup>) for each model:

```r
rbind(
    augment(ps_mod, juice(ps_rec)) |> select(income, .pred, .resid) |> mutate(model = "parallel_slopes"),
    augment(nps_mod, juice(nps_rec)) |> select(income, .pred, .resid) |> mutate(model = "interaction")
)
```

1. Group rows by the `model` variable (use the `group_by` function).
1. Calculate the variance of income over the variance of your predictions for each model using the "grouped" data from the step above (use `summarize(r_squared = var(.pred) / var(income))`).

In [ ]:

fulldata = rbind(
    augment(ps_mod, juice(ps_rec)) |> select(income, .pred, .resid) |> mutate(model = "parallel_slopes"),
    augment(nps_mod, juice(nps_rec)) |> select(income, .pred, .resid) |> mutate(model = "interaction")
)

names(fulldata)

fulldata_r2 = fulldata |> group_by(model) |> summarise(r_squared = var(.pred)/ var(income))
fulldata_r2



🎶 Note: you can also use the `glance` function with a model as input to find the coefficient of determination.

In [ ]:
glance(ps_mod)
glance(nps_mod)

❓Compare the adjusted proportion of variance accounted for in each model. Based on this comparison, which model do you prefer? Why? 

**Answer:**

### The R^2 value is very nearly the same in the two models.  Since we are adding complexity with the interaction term, but aren't getting much improvement, I think the simpler parallel slope model is best.




❓Using your preferred model, based on your regression model parameters (and the data visualizations), is `income` greater in states that have lower or higher levels of `urbanization`? By how much?

**Hint:** use the `tidy` function with your model as input to access the parameters in a nice table.

**Answer:**

In [ ]:
gg
tidy(ps_mod)

#### For the parallel slopes model, the intercept estimate for urbanization_low is -7333, which I interpret to mean that this model shows people earn $7333 less per year, 
####  taking education into account, if they live in a low urbanization state.  The gg graph shows the lines about 7000 dollars apart.




❓For every one percentage point increase of high-school educated adults in a state (`hs` variable), what is the associated average increase in `income`?

**Answer:**

#### For the parallel slopes model, the hs estimate is 1987.  This means that the slope is 1987 dollars per percent hs.  So if you increase 1% in proportion with high school education, the income increases by 1987.




In [ ]:
## RAW R FILE ACTUALLY USED IN VANILLA R:

##################################################
## Complete the exercises below For Assignment #4.
##
##
## Use Markdown cells for the non-code answers.
## 
## In this assignment we will work with the data underlying the FiveThirtyEight article "Higher Rates Of Hate Crimes Are Tied To Income Inequality".
## 
## Load the tidymodels, readr, and moderndive packages in the cell below.
##################################################


## cell 1
install.packages(c('tidymodels','readr','moderndive'))
install.packages('curl') # for download direct from file
library(tidymodels)
library(readr)
library(moderndive)

####################################################
## output 1 
#> library(tidymodels) 
# ── Attaching packages ────────────────────────────────────── tidymodels 1.5.0 ──
# ✔ broom        1.0.13     ✔ recipes      1.3.2 
# ✔ dials        1.4.3      ✔ rsample      1.3.2 
# ✔ dplyr        1.2.1      ✔ tailor       0.1.0 
# ✔ ggplot2      4.0.3      ✔ tidyr        1.3.2 
# ✔ infer        1.1.0      ✔ tune         2.1.0 
# ✔ modeldata    1.5.1      ✔ workflows    1.3.0 
# ✔ parsnip      1.6.0      ✔ workflowsets 1.1.1 
# ✔ purrr        1.2.2      ✔ yardstick    1.4.0 
# ── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
# ✖ purrr::discard() masks scales::discard()
# ✖ dplyr::filter()  masks stats::filter()
# ✖ dplyr::lag()     masks stats::lag()
# ✖ recipes::step()  masks stats::step()
# • Learn how to get started at https://www.tidymodels.org/start/
# 
#> library(readr)
# 
# Attaching package: ‘readr’
# 
# The following object is masked from ‘package:yardstick’:
# 
#     spec
# 
# The following object is masked from ‘package:scales’:
# 
#     col_factor
# 
# > library(moderndive)
#########################################


## cell 2:
# We can read the data from a CSV file at the following URL: http://bit.ly/2ItxYg3.
#
# Use the read_csv function to read the data into our R session. Call the new table hate_crimes.
library(curl)
hate_crimes = read_csv('http://bit.ly/2ItxYg3')
head(hate_crimes)

###################
## output from cell 2:
# > hate_crimes = read_csv('http://bit.ly/2ItxYg3')
# 
# [1mindexed[0m [32m0B[0m in [36m 0s[0m, [32m0B/s[0m
# [1mindexed[0m [32m2.15GB[0m in [36m 0s[0m, [32m2.15GB/s[0m
#                                                                               
# Rows: 51 Columns: 9
# ── Column specification ────────────────────────────────────────────────────────
# Delimiter: ","
# chr (5): state, median_house_inc, trump_support, unemployment, urbanization
# dbl (4): share_pop_metro, hs, hate_crimes, income
# 
# ℹ Use `spec()` to retrieve the full column specification for this data.
# ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
# > head(hate_crimes)
# # A tibble: 6 × 9
#   state        median_house_inc share_pop_metro    hs hate_crimes trump_support
#   <chr>        <chr>                      <dbl> <dbl>       <dbl> <chr>        
# 1 New Mexico   low                         0.69    83       0.295 low          
# 2 Maine        low                         0.54    90       0.616 low          
# 3 New York     low                         0.94    85       0.351 low          
# 4 Illinois     low                         0.9     86       0.195 low          
# 5 Delaware     high                        0.9     87       0.323 low          
# 6 Rhode Island high                        1       85       0.095 low          
# # ℹ 3 more variables: unemployment <chr>, urbanization <chr>, income <dbl>
# 


## Next, let’s add the high-school degree variable into the mix by creating a scatterplot showing:
## 
## x Income on the y-axis (this is the income variable)
## x Percent of adults 25 or older with a high school degree on the x-axis (this is the hs variable)
## x The points colored by level of urbanization in a region (this is the variable urbanization)
## x In addition, add a line of best fit (regression line) for each level of the variable urbanization
## xx (one for “low”, one for “high”).
## 
## Add the regression lines to the plot using the geom_parallel_slopes function from the moderndive 
## package. This function will draw the regression lines based on fitting a regression model 
## with parallel slopes (i.e., with no interaction between hs and urbanization).

## cell 3:

##data.low = hate_crimes %>% filter(urbanization=='low') 
## data.high = hate_crimes %>% filter(urbanization=='high') 


gg = ggplot(data=hate_crimes, aes(x=hs,y=income, group=urbanization)) + geom_point(aes(col=share_pop_metro, shape=urbanization)) + 
  geom_parallel_slopes(show.legend=TRUE) 
gg

# > gg = ggplot(data=hate_crimes, aes(x=hs,y=income, group=urbanization)) + geom_point(aes(col=share_pop_metro)) + 
# +   geom_parallel_slopes() 
# >   # separate models just to compare: geom_smooth(data=data.low, aes(x=hs,y=income), method='lm',se=FALSE)  +
# >   # geom_smooth(data=data.high, aes(x=hs,y=income), method='lm',se=FALSE)  
# > gg
# Warning messages:
# 1: Removed 3 rows containing non-finite outside the scale range (`stat_parallel_slopes()`). 
# 2: Removed 3 rows containing missing values or values outside the scale range (`geom_point()`). 
# 

#❓  Which regression line (high urbanization or low urbanization) appears to have the larger intercept?
##Answer:

## The high household income category seems to have a higher intercept.  
## I added a different symbol for the points so I could be sure which is urbanization high and low.

## Now let’s create a second scatterplot using the same variables, but this time draw the regression lines using geom_smooth(method = "lm"), which will allow for separate, non-parallel slopes for each urbanization group.
## Code your scatter plot in the cell below.

gg2 = ggplot(data=hate_crimes, aes(x=hs,y=income, group=urbanization, col=share_pop_metro, shape=urbanization)) + geom_point() + 
 geom_smooth(method='lm')    
  #geom_parallel_slopes(show.legend=TRUE) 
  # separate models just to compare: geom_smooth(data=data.low, aes(x=hs,y=income), method='lm',se=FALSE)  +
gg2

## ❓ Based on visually comparing the two models shown above, do you think it would be best to run a “parallel slopes” model (i.e. a model that estimates one shared slope for the two levels of urbanization), or a more complex “interaction model” (i.e. a model that estimates a separate slope for the two levels of urbanization)?

## Answer:

##### The models look very similar visually, so I'd go with no separate slope as that is the simpler model.


## Fit the following two regression models that examine the relationship between household income (as response variable), and high-school education (hs) and urbanization as explanatory variables:

## A parallel slopes model (i.e., no interaction between hs and urbanization). ❗️Save the data recipe and model under the variables ps_rec and ps_mod, respectively.
## A non-parallel slopes model (i.e., allow hs and urbanization to interact in your model). ❗️Save the data recipe and model under the variable: nps_rec and nps_mod, respectively.
## Code you your models in the cell below.

# parallel slopes model

ps_rec = recipe(income ~ hs + urbanization, data = hate_crimes) |> 
    step_naomit(everything()) |>   # remove missing values
    step_dummy(urbanization) |>    # dummy encode the urbanization variable
    prep()                         # run the recipe on the training data provided

mod = linear_reg() |> set_engine('lm')

ps_mod = mod |> fit(income ~ ., juice(ps_rec))

ps_mod

# parsnip model object
# 
# 
# Call:
# stats::lm(formula = income ~ ., data = data)
# 
# Coefficients:
#      (Intercept)                hs  urbanization_low  
#          -113725              1987             -7333  

# non-parallel slopes model
# x ❗️ you can use most of the code above, just add step_interact(~ starts_with("urbanization"):hs) to the recipe and
#    switch the variable names...

nps_rec = recipe(income ~ hs + urbanization , data = hate_crimes) |> 
    step_naomit(everything()) |>   # remove missing values
    step_dummy(urbanization) |>    # dummy encode the urbanization variable
    step_interact(terms = ~ starts_with("urbanization"):hs) |>
    prep()                         # run the recipe on the training data provided

nmod = linear_reg() |> set_engine('lm')

nps_mod = nmod |> fit(income ~ ., juice(nps_rec))

nps_mod

# parsnip model object
# 
# 
# Call:
# stats::lm(formula = income ~ ., data = data)
# 
# Coefficients:
#           (Intercept)                     hs       urbanization_low  urbanization_low_x_hs  
#                -95647                   1777                 -35394                    324  

## The following code creates a table of your model predictions over the training data. Calculate the coefficient of determination (R2) for each model:

fulldata = rbind(
    augment(ps_mod, juice(ps_rec)) |> select(income, .pred, .resid) |> mutate(model = "parallel_slopes"),
    augment(nps_mod, juice(nps_rec)) |> select(income, .pred, .resid) |> mutate(model = "interaction")
)
## x Group rows by the model variable (use the group_by function).
## Calculate the variance of income over the variance of your predictions for each model using the "grouped" data from the step above (use summarize(r_squared = var(.pred) / var(income))).

names(fulldata)

fulldata_r2 = fulldata |> group_by(model) |> summarise(r_squared = var(.pred)/ var(income))
fulldata_r2

# 
# # A tibble: 2 × 2
#   model           r_squared
#   <chr>               <dbl>
# 1 interaction         0.575
# 2 parallel_slopes     0.572
# 

# Note: you can also use the glance function with a model as input to find the coefficient of determination.

glance(ps_mod)
glance(nps_mod)

# 
# > glance(ps_mod)
# # A tibble: 1 × 12
#   r.squared adj.r.squared sigma statistic p.value    df logLik   AIC   BIC deviance df.residual  nobs
#       <dbl>         <dbl> <dbl>     <dbl>   <dbl> <dbl>  <dbl> <dbl> <dbl>    <dbl>       <int> <int>
# 1     0.572         0.553 6326.      30.0 5.20e-9     2  -487.  981.  989.   1.80e9          45    48
# > glance(nps_mod)
# # A tibble: 1 × 12
#   r.squared adj.r.squared sigma statistic p.value    df logLik   AIC   BIC deviance df.residual  nobs
#       <dbl>         <dbl> <dbl>     <dbl>   <dbl> <dbl>  <dbl> <dbl> <dbl>    <dbl>       <int> <int>
# 1     0.575         0.546 6374.      19.8 2.81e-8     3  -487.  983.  992.   1.79e9          44    48
# > 
# 

## Compare the adjusted proportion of variance accounted for in each model. Based on this comparison, which model do you prefer? Why?

## Answer:

#### The R^2 value is very nearly the same in the two models.  Since we are adding complexity with the interaction term, but aren't getting much improvement, I think the simpler parallel slope model is best.

## ❓Using your preferred model, based on your regression model parameters (and the data visualizations), is income greater in states that have lower or higher levels of urbanization? By how much?

## Hint: use the tidy function with your model as input to access the parameters in a nice table.

## Answer:
gg
tidy(ps_mod)

#### For the parallel slopes model, the intercept estimate for urbanization_low is -7333, which I interpret to mean that this model shows people earn $7333 less per year, 
####  taking education into account, if they live in a low urbanization state.  The gg graph shows the lines about 7000 dollars apart.

## For every one percentage point increase of high-school educated adults in a state (hs variable), what is the associated average increase in income?

## Answer:

#### For the parallel slopes model, the hs estimate is 1987.  This means that the slope is 1987 $ / percent hs.  So if you increase 1% in proportion with high school education, the income increases by $1987.
